# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, following the [Croissant format](https://mlcommons.org/croissant/).

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
We begin by loading the dataset's schema and examining the dataset-level metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Instantiate the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print summary information
print(f"Dataset name: {getattr(metadata, 'name', None)}\n")
print(f"Description: {getattr(metadata, 'description', None)}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")

## 2. Data Overview
List all available record sets (tables/objects) and their fields within this dataset using their `@id`.

We will display the record sets, their fields, and provide IDs for reference in later steps.

In [ ]:
# Gather all record sets and their fields
record_sets = list(dataset.record_sets)
print(f"Number of record sets in this dataset: {len(record_sets)}\n")
record_sets_summary = []

for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    rs_name = getattr(rs, 'name', None)
    print(f"RecordSet: {rs_name}    (@id: {rs_id})")
    if hasattr(rs, 'fields'):
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    - {getattr(field, 'name', None)} (@id: {getattr(field, '@id', None)})")
    print()
    record_sets_summary.append({'id': rs_id, 'name': rs_name})

if not record_sets:
    print("No record sets found in the top-level metadata.\nIf using mlcroissant>=0.10, you can inspect the first distribution as a record set.")

## 2.1 Fallback: Find Tabular File RecordSets
FAIR^2 schemas often store tabular record sets as datasets linked via `distribution`. Let's list any distributions (files) and their tabular columns.

In [ ]:
# List distributions in metadata
distributions = getattr(metadata, 'distribution', [])
print(f"Number of distributions: {len(distributions)}\n")
dist_info = []
for dist in distributions:
    dist_id = getattr(dist, '@id', None)
    name = getattr(dist, 'name', None)
    content_url = getattr(dist, 'contentUrl', None)
    encoding = getattr(dist, 'encodingFormat', None)
    print(f"Distribution: {name} (@id: {dist_id})")
    print(f"  contentUrl: {content_url}")
    print(f"  encodingFormat: {encoding}")
    if hasattr(dist, 'columns'):
        print("  Columns:")
        for col in dist.columns:
            print(f"    - {getattr(col, 'name', None)} (@id: {getattr(col, '@id', None)})")
    print()
    dist_info.append({'id': dist_id, 'name': name, 'encoding': encoding})

## 3. Data Extraction
Let's extract data into DataFrames. We'll use the first available record set, or if none, the first distribution file as a tabular record set. All IDs are referenced by their `@id`.


In [ ]:
# Fallback for datasets with no explicit recordSet: use distribution @id
if len(record_sets) > 0:
    # Use explicit record sets
    record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]
else:
    # Use the distribution as record set
    record_set_ids = [dist_info[0]['id']] if dist_info else []

print(f"Record set IDs to be loaded: {record_set_ids}\n")

# Load all dataframes by record set id
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for: {rs_id}")
    try:
        df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
        print(f"  Columns: {df.columns.tolist()}")
        dataframes[rs_id] = df
        print(df.head(), "\n")
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}")

# For subsequent cells, pick the main record set id
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
We use a numeric field for filtering, normalization, and grouping. All column accesses are by their `@id`, as shown in the DataFrame above.

In [ ]:
# EDA: Filtering, normalization, grouping
df = dataframes.get(main_record_set_id)
if df is not None:
    print(f"Columns in record set '{main_record_set_id}':\n{list(df.columns)}\n")

    # Try to automatically select a likely numeric column by 'coverage' or 'mw' or first float/int column
    numeric_candidates = [col for col in df.columns if 'coverage' in col.lower() or 'mw' in col.lower() or 'peptide' in col.lower() or df[col].dtype in [int, float]]
    if not numeric_candidates:
        # Try to find numeric columns by type
        numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field for analysis: {numeric_field}\n")
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (by @id):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to pick a grouping field
        group_candidates = [col for col in df.columns if 'sample' in col.lower() or 'type' in col.lower() or 'group' in col.lower() or 'category' in col.lower()]
        group_field = group_candidates[0] if group_candidates else None

        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field} (@id):")
            print(grouped_df.head())
        else:
            print("No obvious grouping field found in columns.")
    else:
        print("No numeric field candidates found in this record set.")
else:
    print("No DataFrame available to perform EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and 'numeric_field' in locals():
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
else:
    print("No numeric field available to plot.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset using `mlcroissant`. We:
- Loaded the Croissant schema and listed available record sets (by `@id`)
- Extracted tabular data into a DataFrame and inspected its columns
- Performed basic EDA: filtered and normalized by a numeric field (using its `@id`), and optionally grouped records
- Visualized a key field's distribution

This structure can be adapted to other Croissant-based datasets; always reference fields by `@id` for maximum reproducibility. For further analyses, consult the dataset's README and Croissant metadata for field semantics and recommended usage.